### Query STAC API

In [1]:
import pystac
import pystac_client

catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")

california_bbox = [-124.4096, 32.5343, -114.1312, 42.0095]

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=california_bbox,
    datetime="2025-10-01/2025-11-01",
)

items = list(search.items())
items[0]

<Item id=LC09_L2SP_046033_20251101_02_T1>

### Create Zarr store and metadata arrays

In [2]:
import numpy as np
import zarr
from zarr.dtype import VariableLengthBytes, VariableLengthUTF8

root = zarr.open_group("spatial_index.zarr", mode="w", zarr_format=3)
meta = root.create_group("meta")

In [3]:
meta.create_array(
    "date",
    shape=(0,),
    dtype="datetime64[ms]",
)
meta.create_array(
    "bbox",
    shape=(0,),
    dtype=VariableLengthBytes(),
)
meta.create_array(
    "cloud_cover",
    shape=(0,),
    dtype="float32"
)
meta.create_array(
    "id",
    shape=(0,),
    dtype=VariableLengthUTF8(),
)

/Users/seanharkins/projects/zarr-datafusion-internal/.venv/lib/python3.12/site-packages/zarr/core/dtype/npy/bytes.py:1143: UnstableSpecificationWarning: The data type (VariableLengthBytes()) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


<Array file://spatial_index.zarr/meta/id shape=(0,) dtype=StringDType()>

### Write zarr-datafusion-search metadata for each of the STAC items

In [4]:
import shapely
def write_item_metadata(item: pystac.Item, meta: zarr.Group):
    dt = meta["date"]
    dt.append(np.array([item.datetime], dtype="datetime64[ms]"))

    bbox_wkb = shapely.to_wkb(shapely.box(*item.bbox))
    bbox = meta["bbox"]
    bbox.append(np.array([bbox_wkb]))
    
    cloud_cover = meta["cloud_cover"]
    cloud_cover.append(np.array([item.properties["eo:cloud_cover"]]))

    id = meta["id"]
    id.append(np.array([f"{item.id}"]))

In [5]:
for item in items:
    write_item_metadata(item=item, meta=meta)

/var/folders/mp/33cxt8xj36bbxj0jqdwbfwyc0000gn/T/ipykernel_89401/337541504.py:4: UserWarning: no explicit representation of timezones available for np.datetime64
  dt.append(np.array([item.datetime], dtype="datetime64[ms]"))


### Register Zarr store with custom TableProvider

In [6]:
from datafusion import SessionContext
from obstore.store import LocalStore
from zarr_datafusion_search import ZarrTable

store = LocalStore("spatial_index.zarr")
zarr_table = await ZarrTable.from_obstore(store, "/meta")

ctx = SessionContext()
ctx.register_table("spatial_index", zarr_table)

/var/folders/mp/33cxt8xj36bbxj0jqdwbfwyc0000gn/T/ipykernel_89401/4286122550.py:6: RuntimeWarning: Successfully reconstructed a store defined in another Python module. Connection pooling will not be shared across store instances.
  zarr_table = await ZarrTable.from_obstore(store, "/meta")


### Do a spatial query with no spatial index

In [12]:
from geodatafusion import register_all

register_all(ctx)

df = ctx.sql("""
    SELECT id, date
    FROM spatial_index
    WHERE ST_Intersects(
        bbox,
        ST_GeomFromText('POLYGON((-118.5 33.9, -118.1 33.9, -118.1 34.2, -118.5 34.2, -118.5 33.9))')
    )
    ORDER BY id
""")
df.show()

DataFrame()
+---------------------------------+-------------------------+
| id                              | date                    |
+---------------------------------+-------------------------+
| LC08_L2SP_040037_20251014_02_T1 | 2025-10-14T18:22:49.508 |
| LC08_L2SP_040037_20251030_02_T1 | 2025-10-30T18:22:50.552 |
| LC08_L2SP_041036_20251005_02_T1 | 2025-10-05T18:28:34.608 |
| LC08_L2SP_041036_20251021_02_T1 | 2025-10-21T18:28:33.420 |
| LC08_L2SP_041037_20251005_02_T1 | 2025-10-05T18:28:58.499 |
| LC08_L2SP_041037_20251021_02_T1 | 2025-10-21T18:28:57.315 |
| LC09_L2SP_040037_20251006_02_T1 | 2025-10-06T18:22:51.880 |
| LC09_L2SP_040037_20251022_02_T1 | 2025-10-22T18:22:54.355 |
| LC09_L2SP_041036_20251013_02_T1 | 2025-10-13T18:28:42.584 |
| LC09_L2SP_041036_20251029_02_T1 | 2025-10-29T18:28:35.185 |
| LC09_L2SP_041037_20251013_02_T1 | 2025-10-13T18:29:06.475 |
| LC09_L2SP_041037_20251029_02_T1 | 2025-10-29T18:28:59.076 |
+---------------------------------+-----------------------

### Create a spatial index and store it in the _indexes_ group

In [8]:
from geoindex_rs import rtree as rt

geoms = shapely.from_wkb(meta["bbox"][:])  # vectorized, returns ndarray of geometries
bounds = shapely.bounds(geoms)
min_x = np.ascontiguousarray(bounds[:, 0])
min_y = np.ascontiguousarray(bounds[:, 1])
max_x = np.ascontiguousarray(bounds[:, 2])
max_y = np.ascontiguousarray(bounds[:, 3])

builder = rt.RTreeBuilder(num_items=len(meta["bbox"][:]))
builder.add(min_x, min_y, max_x, max_y)
tree = builder.finish()

In [9]:
indexes = root.create_group("indexes")
tree_bytes = bytes(tree)
indexes.create_array(
    "bbox",
    data=np.frombuffer(tree_bytes, dtype=np.uint8)
)

<Array file://spatial_index.zarr/indexes/bbox shape=(7488,) dtype=uint8>

### The query returns the same result when using the index

In [11]:
df = ctx.sql("""
    SELECT id, date
    FROM spatial_index
    WHERE ST_Intersects(
        bbox,
        ST_GeomFromText('POLYGON((-118.5 33.9, -118.1 33.9, -118.1 34.2, -118.5 34.2, -118.5 33.9))')
    )
    ORDER BY id
""")
df.show()

DataFrame()
+---------------------------------+-------------------------+
| id                              | date                    |
+---------------------------------+-------------------------+
| LC08_L2SP_040037_20251014_02_T1 | 2025-10-14T18:22:49.508 |
| LC08_L2SP_040037_20251030_02_T1 | 2025-10-30T18:22:50.552 |
| LC08_L2SP_041036_20251005_02_T1 | 2025-10-05T18:28:34.608 |
| LC08_L2SP_041036_20251021_02_T1 | 2025-10-21T18:28:33.420 |
| LC08_L2SP_041037_20251005_02_T1 | 2025-10-05T18:28:58.499 |
| LC08_L2SP_041037_20251021_02_T1 | 2025-10-21T18:28:57.315 |
| LC09_L2SP_040037_20251006_02_T1 | 2025-10-06T18:22:51.880 |
| LC09_L2SP_040037_20251022_02_T1 | 2025-10-22T18:22:54.355 |
| LC09_L2SP_041036_20251013_02_T1 | 2025-10-13T18:28:42.584 |
| LC09_L2SP_041036_20251029_02_T1 | 2025-10-29T18:28:35.185 |
| LC09_L2SP_041037_20251013_02_T1 | 2025-10-13T18:29:06.475 |
| LC09_L2SP_041037_20251029_02_T1 | 2025-10-29T18:28:59.076 |
+---------------------------------+-----------------------